In [1]:
import pandas as pd
import numpy as np
import cv2 as cv
from pathlib import Path 
import seaborn as sns
import matplotlib.pyplot as plt
from StatTools.generators.ndfnoise_generator import ndfnoise
from tqdm import tqdm
import plotly.express as px
import os
import warnings 
def draw_traj(trajectories, frame_shape: tuple, thickness: int):
    
    ants_num = trajectories.shape[1]
    bgs = [np.zeros(shape=frame_shape, dtype=np.uint8) for _ in range(ants_num)]
    
    for ant_idx in range(ants_num):
        
        traj_raw = trajectories[:, ant_idx, :]
        
        valid_mask = ~np.any(np.isnan(traj_raw), axis=1)
        traj_clean = traj_raw[valid_mask]
        
        if len(traj_clean) < 2:
            continue
         
        traj_int = np.ascontiguousarray(traj_clean.astype(np.int32))
        
        cv.polylines(bgs[ant_idx], [traj_int], isClosed=False, color=255, thickness=thickness)
    
    return bgs


def calc_iou(tracks, thickness,  frame_shape):
    bgs = draw_traj(tracks, frame_shape=frame_shape, thickness = thickness)
    bgs_arr_bool = np.stack(bgs).astype(np.bool)
    union = bgs_arr_bool.sum(axis=0)
    union_mask = union.astype(np.bool)
    atleast_two_track_intersection = union - union_mask
    iou = atleast_two_track_intersection.sum() / union.sum()
    return iou, atleast_two_track_intersection, union


def gen_traj(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
             hurst_move: float = 0.5, hurst_species: float = 0.5, 
             start_point = None):
    

    dx = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)
    dy = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(margin, frame_shape[1] - margin, (ants_num,))
        start_y = np.random.randint(margin, frame_shape[0] - margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

In [2]:
# dir_path = Path(r'E:\asp\Ants\P.yeensis\nmot_tracks')

In [3]:
# df_test = pd.read_csv(next(dir_path.glob("*.csv")))
# df_test = df_test.astype({'x':np.int32, 'y':np.int32})
# x_wide = df_test.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
# y_wide = df_test.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
# trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)

In [4]:
# lens = np.exp(np.arange(np.log(10), np.log(trajectories.shape[0]), 0.5)).astype(np.int32)
# _result1 = [] 
# for length in lens:
#     track_slice = trajectories[:length]
#     iou, _, _  = calc_iou(track_slice, 10, (500, 900))

#     _result1.append({
#                     'length':length,
#                     'iou': iou,
#                 })

In [5]:
species_list = ['F.rufa', 'P.yeensis']
result = []
for species in species_list:
    dir_path = Path(fr'E:\asp\Ants\{species}\nmot_tracks')
    pbar = tqdm(dir_path.glob("*.csv"))
    for file_path in pbar:
        pbar.set_description(f"Processing {file_path.name}")
        df_test = pd.read_csv(file_path)
        df_test = df_test.astype({'x':np.int32, 'y':np.int32})
        x_wide = df_test.pivot_table(columns='track_id', index='frame', values='x', fill_value=np.nan)
        y_wide = df_test.pivot_table(columns='track_id', index='frame', values='y', fill_value=np.nan)
        trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)

        lens = np.exp(np.arange(np.log(10), np.log(trajectories.shape[0]), 0.5)).astype(np.int32)
        
        try:
            for length in lens:
                track_slice = trajectories[:length]
                iou, _, _  = calc_iou(track_slice, 10, (500, 900))

                result.append({'species': species,
                                'name': file_path.name,
                                'length':length,
                                'iou': iou,
                            })
        except Exception as e:
            warnings.warn(f"Ошибка в {file_path.name}: {e}")
            continue

Processing S1240025.csv: : 21it [1:43:35, 295.99s/it]          
Processing S2190017.csv: : 56it [1:09:11, 74.14s/it]   


In [6]:
df_res = pd.DataFrame(result)

In [7]:
df_res.to_pickle('iou_estimation.pkl')

In [10]:
px.scatter(df_res, x='length', y='iou', color='species')